In [1]:
import numpy as np
import pandas as pd
from itertools import product

In [2]:
# ==========================================================
# PARAMETERS
# ==========================================================
m = 1.0
k = 1.5
c = 0.5

# tau = 0.5
# sigma = np.sqrt(2)

# ==========================================================
# CASES
# ==========================================================
# Parameter values
tau_values = [0.05, 0.5, 2.0]
sigma_values = [2.0, np.sqrt(2), 1.0]
# Create all 9 combinations
cases = {}
for i, (tau, sigma) in enumerate(product(tau_values, sigma_values), start=1):
    cases[f"case_{i}"] = {
        "tau": tau,
        "sigma": sigma
    }
# Print dictionary
for name, params in cases.items():
    print(name, params)
# cases = {
#     "case_1": {"tau": 0.05, "sigma": 2.0},
#     "case_2": {"tau": 0.5,  "sigma": np.sqrt(2)},
#     "case_3": {"tau": 2.0,  "sigma": 1.0} 
# }

dt = 0.001
t_max = 50.0

x0 = 1.0
v0 = 0.0
eta0 = 0.0

n_runs = 50
base_seed = 42

case_1 {'tau': 0.05, 'sigma': 2.0}
case_2 {'tau': 0.05, 'sigma': 1.4142135623730951}
case_3 {'tau': 0.05, 'sigma': 1.0}
case_4 {'tau': 0.5, 'sigma': 2.0}
case_5 {'tau': 0.5, 'sigma': 1.4142135623730951}
case_6 {'tau': 0.5, 'sigma': 1.0}
case_7 {'tau': 2.0, 'sigma': 2.0}
case_8 {'tau': 2.0, 'sigma': 1.4142135623730951}
case_9 {'tau': 2.0, 'sigma': 1.0}


In [3]:
# ==========================================================
# TIME GRID
# ==========================================================
# time = np.arange(0, t_max + dt, dt)
time = np.linspace(0, t_max, int(t_max/dt)+1)

In [4]:
# ==========================================================
# SINGLE RUN SOLVER
# ==========================================================
def solve_system(time, m, k, c, tau, sigma, x0, v0, eta0, seed):
    """
    One realization of damped harmonic oscillator with OU noise.
    """

    rng = np.random.default_rng(seed)

    n = len(time)
    dt = time[1] - time[0]

    x = np.zeros(n)
    v = np.zeros(n)
    # a = np.zeros(n)
    eta = np.zeros(n)

    x[0] = x0
    v[0] = v0
    eta[0] = eta0

    for i in range(n - 1):

        # total force
        force = -k * x[i] - c * v[i] + eta[i]

        # acceleration
        # a[i] = force / m
        a = force / m

        # update motion
        # v[i + 1] = v[i] + a[i] * dt
        v[i + 1] = v[i] + a * dt
        x[i + 1] = x[i] + v[i] * dt

        # OU noise update
        dW = np.sqrt(dt) * rng.normal()

        eta[i + 1] = (
            eta[i]
            + (-eta[i] / tau) * dt
            + sigma * np.sqrt(2 / tau) * dW
        )

    # final acceleration
    # a[-1] = (-k * x[-1] - c * v[-1] + eta[-1]) / m

    return x, v, eta

In [5]:
# ==========================================================
# DERIVED QUANTITIES
# ==========================================================
def compute_forces(x, v, eta, k, c):
    f_spring = -k * x
    f_damp = -c * v
    f_total = f_spring + f_damp + eta

    return f_spring, f_damp, f_total


def compute_energies(x, v, m, k):
    ke = 0.5 * m * v**2
    pe = 0.5 * k * x**2
    te = ke + pe

    return ke, pe, te

In [6]:
# ==========================================================
# LOOP OVER CASES
# ==========================================================
for case_name, params in cases.items():

    tau = params["tau"]
    sigma = params["sigma"]

    print(f"Running {case_name} : tau={tau}, sigma={sigma}")
    
    # ==========================================================
    # STORAGE ARRAYS
    # ==========================================================
    n_time = len(time)

    all_x = np.zeros((n_runs, n_time))
    all_v = np.zeros((n_runs, n_time))
    # all_a = np.zeros((n_runs, n_time))
    all_eta = np.zeros((n_runs, n_time))
    
    # ==========================================================
    # ENSEMBLE LOOP
    # ==========================================================
    for run in range(n_runs):

        seed = base_seed + run

        x, v, eta = solve_system(time, m, k, c, tau, sigma, x0, v0, eta0, seed)

        all_x[run] = x
        all_v[run] = v
        # all_a[run] = a
        all_eta[run] = eta

    # ==========================================================
    # ENSEMBLE STATISTICS
    # ==========================================================
    mean_x = np.mean(all_x, axis=0)
    var_x  = np.var(all_x, axis=0)
    
    mean_v = np.mean(all_v, axis=0)
    var_v  = np.var(all_v, axis=0)
    
    # mean_a = np.mean(all_a, axis=0)
    # var_a  = np.var(all_a, axis=0)
    
    mean_eta = np.mean(all_eta, axis=0)
    var_eta  = np.var(all_eta, axis=0)
    
    
    # ---------- Derived quantities ----------
    mean_f_spring = -k * mean_x
    mean_f_damp   = -c * mean_v
    mean_f_total  = mean_f_spring + mean_f_damp + mean_eta
    
    mean_a = mean_f_total / m
    
    mean_ke = 0.5 * m * np.mean(all_v**2, axis=0)
    mean_pe = 0.5 * k * np.mean(all_x**2, axis=0)
    # mean_te = mean_ke + mean_pe
    mean_te = np.mean(0.5*k*all_x**2 + 0.5*m*all_v**2, axis=0)

    # ==========================================================
    # SAVE CSV (summary statistics)
    # ==========================================================
    summary = pd.DataFrame({
        "time": time,

        "mean_x": mean_x,
        "var_x": var_x,

        "mean_v": mean_v,
        "var_v": var_v,

        "mean_eta": mean_eta,
        "var_eta": var_eta,

        "mean_a": mean_a,

        "mean_f_spring": mean_f_spring,
        "mean_f_damp": mean_f_damp,
        "mean_f_total": mean_f_total,

        "mean_ke": mean_ke,
        "mean_pe": mean_pe,
        "mean_te": mean_te
    })

    csv_name = f"ensemble_{case_name}.csv"
    summary.to_csv(csv_name, index=False)

    # ==========================================================
    # SAVE NPZ (raw trajectories + metadata)
    # ==========================================================
    npz_name = f"raw_{case_name}.npz"

    np.savez(
        npz_name,

        time=time,

        all_x=all_x,
        all_v=all_v,
        all_eta=all_eta,

        m=m,
        k=k,
        c=c,

        tau=tau,
        sigma=sigma,

        dt=dt,
        t_max=t_max,

        x0=x0,
        v0=v0,
        eta0=eta0,

        n_runs=n_runs,
        base_seed=base_seed
    )

    print(f"Saved CSV : {csv_name}")
    # print(f"Saved NPZ : {npz_name}")
    print("-" * 50)

Running case_1 : tau=0.05, sigma=2.0
Saved CSV : ensemble_case_1.csv
Saved NPZ : raw_case_1.npz
--------------------------------------------------
Running case_2 : tau=0.05, sigma=1.4142135623730951
Saved CSV : ensemble_case_2.csv
Saved NPZ : raw_case_2.npz
--------------------------------------------------
Running case_3 : tau=0.05, sigma=1.0
Saved CSV : ensemble_case_3.csv
Saved NPZ : raw_case_3.npz
--------------------------------------------------
Running case_4 : tau=0.5, sigma=2.0
Saved CSV : ensemble_case_4.csv
Saved NPZ : raw_case_4.npz
--------------------------------------------------
Running case_5 : tau=0.5, sigma=1.4142135623730951
Saved CSV : ensemble_case_5.csv
Saved NPZ : raw_case_5.npz
--------------------------------------------------
Running case_6 : tau=0.5, sigma=1.0
Saved CSV : ensemble_case_6.csv
Saved NPZ : raw_case_6.npz
--------------------------------------------------
Running case_7 : tau=2.0, sigma=2.0
Saved CSV : ensemble_case_7.csv
Saved NPZ : raw_case_7.